# NDVI Forest Timelapse — Walkthrough

Build an annotated Sentinel-2 **NDVI timelapse** (MP4 + GIF) over a forest area,
step by step, using the `gee_animation` package.

Two areas of interest drive the run:

- a **frame** rectangle — the animation extent (aspect ratio preserved), and
- a **region** polygon — the important area; only scenes with < `region_max_cloud_percent`
  cloud cover *over this polygon* are kept, and its outline is drawn on each frame.

**Prerequisites** (run once, in a terminal, with the `GEE_animation` conda env active):

```bash
conda activate GEE_animation
pip install -e ".[dev,notebook]"
earthengine authenticate      # one-time Google Earth Engine login
```

> Run this notebook with the **`GEE_animation`** environment as the kernel. It makes
> **live Earth Engine calls** (network + a configured EE project — `hnee-331218`).

## 1. Imports & authenticate

In [ ]:
from gee_animation import auth, aoi, collection, compositing, render
from gee_animation.imaging import colorize
from gee_animation.config import RunConfig

auth.init("hnee-331218")   # ee.Initialize(project=...); prompts to auth if needed
print("Earth Engine initialised.")

## 2. Parameters

Everything about a run lives in a `RunConfig`. Edit any field and re-run from here
down. `ndvi.min`/`max` + `palette` are applied identically to every frame so colour
is comparable across the animation.

In [ ]:
from pathlib import Path

def _repo_path(rel):
    """Resolve a repo-relative path whether the kernel CWD is the repo root or notebooks/."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / rel).exists():
            return str(base / rel)
    return rel

cfg = RunConfig(
    name="wne_ndvi",                                # -> out/wne_ndvi.{mp4,gif}
    project="hnee-331218",
    frame_aoi={"bbox": [13.82875, 52.94633, 13.96356, 53.02769]},   # WNE frame extent
    region_aoi={"geojson": _repo_path("docs/aoi/wne/wne.geojson")},  # Grumsin polygon (438 pts)
    region_max_cloud_percent=10,   # keep scenes with < 10% cloud over the region
    start="2022-05-01",            # inclusive
    end="2022-08-01",              # exclusive
    sensor="sentinel2",
    cadence="monthly",
    max_cloud_percent=60,          # coarse scene-level pre-filter
    ndvi_min=-0.2, ndvi_max=0.9,
    palette=["#a1622f", "#e8d9a0", "#3b7a2a"],      # brown -> tan -> green
    fps=4, scale=20, dimensions=768,
    draw_region=True,              # draw the region polygon outline on each frame
)
cfg

## 3. Areas of interest

Parse both AOIs into Earth Engine geometries.

In [ ]:
frame = aoi.parse(cfg.frame_aoi)     # rectangle -> render extent + scene bounds
region = aoi.parse(cfg.region_aoi)   # polygon  -> cloud filter + overlay
print("frame bounds:", frame.bounds().getInfo()["coordinates"])
print("region area (km^2):", round(region.area().getInfo() / 1e6, 2))

## 4. Build the cloud-masked NDVI collection

Filter Sentinel-2 to the frame, keep scenes with < `region_max_cloud_percent` cloud
*over the region* (computed on the unmasked image), mask clouds, and add an `NDVI` band.

In [ ]:
coll = collection.build(cfg, frame, region)
n_scenes = coll.size().getInfo()
print(f"{n_scenes} Sentinel-2 scenes pass the frame + region-cloud filter")

## 5. Monthly median frames

Reduce each month to a cloud-robust median NDVI image (empty months skipped).

In [ ]:
frames = compositing.monthly_median(coll, cfg)
print(f"{len(frames)} monthly frames:")
print(", ".join(f.label for f in frames))

## 6. Preview a single frame

Render just the first frame — including the region outline — to check AOI, palette
and cloud masking before committing to the full animation.

In [ ]:
from PIL import Image
from IPython.display import display

frame_obj = frames[0]
# render._fetch_thumbnail is the package's internal EE download; used here for a preview.
ndvi_arr, valid = render._fetch_thumbnail(frame_obj.image, cfg, frame)
rgb = colorize(ndvi_arr, cfg.ndvi_min, cfg.ndvi_max, cfg.palette)
rgb = render.apply_nodata(rgb, valid)      # cloud / no-data -> neutral grey
rgb = render.annotate(rgb, frame_obj.label)
rgb = render.add_colorbar(rgb, cfg)
if cfg.draw_region:                        # overlay the region polygon outline
    bounds = render._aoi_bounds(cfg.frame_aoi)
    rings = render._region_rings(cfg.region_aoi)
    rgb = render.draw_region(rgb, bounds, rings)
print(f"Preview of {frame_obj.label}:")
display(Image.fromarray(rgb))

## 7. Render the animation

Run the full pipeline over every frame (region outline drawn automatically when
`cfg.draw_region` is set) and write `out/<name>.mp4` and `out/<name>.gif`.

In [ ]:
paths = render.render(frames, cfg, geometry=frame)
paths

## 8. Show the result

In [ ]:
from IPython.display import Image as IPyImage

gif = next(p for p in paths if p.suffix == ".gif")
print(f"{gif}")
IPyImage(filename=str(gif))

In [ ]:
from IPython.display import Video

mp4 = next((p for p in paths if p.suffix == ".mp4"), None)
Video(str(mp4), embed=True) if mp4 else "MP4 not produced (ffmpeg unavailable) - see GIF above."